# Bulk RNA-seq Differential Expression Pipeline

> **Note:** This pipeline is independent of the scRNA-seq pipeline. Run it in parallel or first — it does not depend on scRNA-seq output and the results are used to validate the single-cell findings.

## The problem this step solves

The 204,883 single-cell atlas gives you per-cell resolution, but single-cell measurements are noisy. Each cell is sequenced at shallow depth (~5,000–10,000 genes detected per cell vs. ~15,000 in bulk). Rare genes may not be detected in any given cell even if the cell expresses them. Statistical power for differential expression is limited when each animal contributes thousands of individual but shallow measurements.

Bulk RNA-seq solves this differently: all cells in the tissue are pooled and sequenced together at high depth. You get one measurement per animal per tissue, but each measurement covers the complete transcriptome at high accuracy. The tradeoff: you lose cell-type resolution (you see the tissue average), but you gain sensitivity for low-expression genes and statistical power from clean biological replicates.

**The two approaches are not competing — they are complementary.** This is exactly how Yang et al. used them:
- Bulk identifies DEGs with high confidence (1,386 unique DEGs across three tissues)
- scRNA-seq maps those DEGs to specific cell types (ASCs drive most of the vWAT response)
- Genes that appear in both provide the strongest evidence — validated by independent methods

## Why this matters for the Yang et al. paper

The paper's bulk results (Figure 2, Table 1) established the tissue-level transcriptional landscape before diving into single-cell resolution. Key findings:

- **1,386 unique DEGs** across three tissues: 568 in scWAT, 562 in vWAT, 256 in SkM
- **94–95% anti-correlation** in adipose tissue: genes upregulated by obesity are downregulated by training and vice versa — exercise has nearly opposite effects to HFD at the transcriptome level
- **Training upregulated** circadian rhythm genes (Dbp, Tef, Nr1d2, Per3) and fatty acid oxidation genes
- **Training downregulated** ECM remodeling genes (Thbs1, Sparc) and inflammatory modules
- **SkM rescue > training**: skeletal muscle showed more DEGs in TH vs. SH (rescue, n=203) than TC vs. SC (training, n=74) — exercise has a stronger transcriptional effect on obese muscle than healthy muscle

These bulk results were then **validated and refined** by the single-cell atlas: the ECM and circadian rhythm gene programs found in bulk turn out to be driven primarily by MSCs, not the bulk tissue average.

## The three contrasts

Every tissue is analyzed with three comparisons that map to specific biological questions:

| Contrast | Comparison | Question |
|----------|-----------|----------|
| Obesity | SH vs. SC | What does HFD do to gene expression? |
| Training | TC vs. SC | What does exercise do on a healthy diet? |
| Rescue | TH vs. SH | Can exercise reverse obesity-induced gene changes? |

The rescue contrast (TH vs. SH) is the most therapeutically relevant — it directly tests whether exercise training can normalize the transcriptome of an obese animal.

## What this notebook does

For each of three tissues (scWAT, vWAT, SkM):
1. Filter to protein-coding genes and deduplicate by gene symbol
2. Build sample metadata parsing the structured sample names
3. Quality assessment: VST normalization, sample distance heatmap, PCA
4. Run DESeq2 for all three contrasts with LFC shrinkage
5. Save significant DE gene tables and summary statistics

> **ML note:** DESeq2 fits a negative binomial GLM per gene — it models the count distribution correctly (overdispersed, not Poisson) and accounts for the mean-variance relationship that makes RNA-seq counts harder to test than continuous measurements. LFC shrinkage (ashr) is regularization applied to fold change estimates, pulling noisy estimates for low-count genes toward zero — equivalent to L1/L2 regularization on regression coefficients.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import scipy.cluster.hierarchy as sch
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D
from pathlib import Path

# pydeseq2: Python port of DESeq2
# pip install pydeseq2
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

## Color palette

In [ ]:
annotation_colors = {
    "diet":     {"HFD": "#E41A1C", "Chow": "#377EB8"},
    "training": {"Training": "#4DAF4A", "Sedentary": "#984EA3"},
    "tissue":   {"SkM": "#1B9E77", "scWAT": "#D95F02", "vWAT": "#7570B3"},
}
hm_colors = sns.color_palette("Blues_r", 255)

## Load and filter the count matrix

The input file `bulk_sample_level_data.txt` is a pre-aggregated count matrix with rows as genes (annotated with Ensembl IDs, MGI symbols, and gene type) and columns as samples.

**Protein-coding filter:** only protein-coding genes are retained for DE analysis. Non-coding RNAs, pseudogenes, and other biotypes are excluded because:
- Their expression quantification is less reliable (many overlap coding gene loci)
- Pathway databases (KEGG, GO) are built around protein-coding genes
- They add noise without interpretable downstream signal for most metabolic questions

**Deduplication by MGI symbol:** some genes have multiple Ensembl IDs (e.g., due to alternative loci or gene model updates). We keep one row per gene symbol, dropping duplicates to ensure a clean 1-to-1 mapping between gene names and count rows.

In [ ]:
bulk_data = pd.read_csv("bulk_sample_level_data.txt", sep="\t")
print(f"Raw dimensions: {bulk_data.shape}")

# Filter to protein-coding genes
pc_mask = bulk_data["Gene type"] == "protein_coding"
bulk_pc = bulk_data[pc_mask].copy()

# Keep only count columns ("intCt" suffix) and the gene symbol column
count_cols = [c for c in bulk_pc.columns if "intCt" in c]
bulk_pc    = bulk_pc[["MGI symbol"] + count_cols].copy()

# Deduplicate by MGI symbol: drop rows with missing or duplicate gene symbols
bulk_pc = bulk_pc.dropna(subset=["MGI symbol"])
bulk_pc = bulk_pc[~bulk_pc["MGI symbol"].duplicated()]
bulk_pc = bulk_pc.set_index("MGI symbol")

# Clean column names: strip the "intCt" suffix to get plain sample names
bulk_pc.columns = bulk_pc.columns.str.replace(".intCt", "", regex=False)

print(f"Protein-coding, deduplicated: {bulk_pc.shape[0]:,} genes x {bulk_pc.shape[1]} samples")
bulk_pc.iloc[:5, :5]

## Build sample metadata

Sample names are structured as `<pheno_class><replicate>.<tissue>` (e.g., `SC1.scWAT`, `TH2.SkM`). We parse this to extract:
- `tissue` — scWAT, vWAT, or SkM (from the part after the dot)
- `pheno_class` — the two-character experimental group code (SC, TC, SH, TH)
- `replicate` — the replicate number within the group
- `diet` and `training` — derived from pheno_class

The `pheno_class` is treated as a factor with a defined level order (`SC < TC < SH < TH`). DESeq2 uses SC as the reference level by default, producing contrasts relative to standard chow sedentary.

In [ ]:
coldata = pd.DataFrame({"sample": bulk_pc.columns})

# Parse tissue and pheno_class from sample name structure: <pheno><rep>.<tissue>
coldata["tissue"]      = coldata["sample"].str.split(".").str[1]
coldata["pheno_class"] = coldata["sample"].str.split(".").str[0].str[:2]
coldata["rep"]         = coldata["sample"].str.split(".").str[0].str[2:]

# Assign biological variables from pheno_class
coldata["diet"]     = coldata["pheno_class"].map(
    {"SC": "Chow", "TC": "Chow", "SH": "HFD", "TH": "HFD"}
)
coldata["training"] = coldata["pheno_class"].map(
    {"SC": "Sedentary", "TC": "Training", "SH": "Sedentary", "TH": "Training"}
)
coldata["pheno_class"] = pd.Categorical(
    coldata["pheno_class"], categories=["SC", "TC", "SH", "TH"]
)
coldata = coldata.set_index("sample")

# Add a replicate index within each tissue x pheno_class group
coldata["rep_recode"] = coldata.groupby(["tissue", "pheno_class"]).cumcount() + 1

print("Sample metadata:")
print(coldata.groupby(["tissue", "pheno_class"]).size().unstack())

## Per-tissue DESeq2 analysis loop

The analysis is run independently for each tissue. The three key analysis stages per tissue are:

### Stage 1 — Quality assessment (VST + PCA + sample distance heatmap)

**VST (Variance Stabilizing Transformation):** DESeq2's VST removes the mean-variance dependence in count data. Raw counts have variance that scales with the mean (Poisson/negative binomial behavior), which would cause highly expressed genes to dominate PCA and distance calculations. VST produces approximately homoskedastic values suitable for Euclidean-distance-based methods.

**Sample distance heatmap:** hierarchical clustering of samples by Euclidean distance in VST space, using Ward's linkage. This is the bulk equivalent of the pseudobulk clustering in Step 4 — it confirms that samples cluster by biological condition rather than technical factors. In this study, diet typically dominates over exercise at the sample-level clustering.

**PCA:** the top 500 most variable genes (by row variance) are used for PCA. Restricting to variable genes removes the majority of uninformative genes and focuses the decomposition on the genes that actually distinguish conditions. PC1 and PC2 are plotted colored by pheno_class; PC3 is available for 3D inspection.

### Stage 2 — Differential expression

**DESeq2** fits a negative binomial generalized linear model per gene with the design `~ pheno_class`. The negative binomial distribution models count overdispersion — the variance of RNA-seq counts is typically greater than the mean (unlike the Poisson distribution assumed by earlier methods).

**Low-count filter:** genes with fewer than 10 total counts across all samples are dropped before fitting. These genes have insufficient data to estimate dispersion reliably and add noise to multiple testing correction.

**IHW (Independent Hypothesis Weighting):** replaces standard Benjamini-Hochberg FDR correction. IHW groups hypotheses by a covariate (here, mean expression) and assigns higher weights to groups with more statistical power. This is more powerful than BH while still controlling FDR — equivalent to the `filterFun = ihw` argument in the R code. In Python, we approximate this with adaptive p-value weighting based on expression level.

**LFC shrinkage (ashr):** raw log2 fold change estimates for low-count genes are extremely noisy (a gene with 1 count in one group and 2 in another appears to double, but this is meaningless noise). The `ashr` shrinkage method (Stephens 2016) pulls these unreliable estimates toward zero based on an empirical prior, making the MA plot and ranked gene lists more interpretable.

### Stage 3 — Three contrasts

Three comparisons are tested per tissue:
1. **TC vs. SC** — effect of exercise training on a healthy diet background
2. **SH vs. SC** — effect of HFD in sedentary animals (obesity model)
3. **TH vs. SH** — effect of exercise training in obese animals (the therapeutic contrast)

Contrasts 1 and 2 come from the default DESeq2 model (reference = SC). Contrast 3 requires releveling to SH as the reference and re-running the Wald test.

In [ ]:
results_store = {}

for tissue in ["scWAT", "vWAT", "SkM"]:
    print(f"\n{'='*60}")
    print(f"Tissue: {tissue}")
    print(f"{'='*60}")

    # ── Subset to this tissue ──────────────────────────────────────
    tissue_samples = coldata[coldata["tissue"] == tissue].index.tolist()
    cts_tissue     = bulk_pc[tissue_samples].astype(int)
    meta_tissue    = coldata.loc[tissue_samples].copy()

    print(f"Samples: {len(tissue_samples)}, Genes (pre-filter): {cts_tissue.shape[0]:,}")

    # ── Low-count filter ───────────────────────────────────────────
    # Genes with <10 total counts across all samples are uninformative
    # and cannot have dispersion estimated reliably.
    keep = cts_tissue.sum(axis=1) >= 10
    cts_tissue = cts_tissue[keep]
    print(f"Genes after low-count filter (sum >= 10): {cts_tissue.shape[0]:,}")

    # ── Stage 1: Quality assessment ───────────────────────────────
    # VST: use log1p of CPM as a lightweight approximation of DESeq2's VST.
    # For a faithful VST, use rpy2 or compute via pydeseq2 internals.
    cpm = cts_tissue.divide(cts_tissue.sum(axis=0), axis=1) * 1e6
    vst = np.log1p(cpm)   # genes x samples

    # Sample distance heatmap
    # Euclidean distance in VST space, Ward's linkage
    from scipy.spatial.distance import pdist, squareform
    sample_dists   = squareform(pdist(vst.T.values, metric="euclidean"))
    dist_df        = pd.DataFrame(sample_dists,
                                  index=tissue_samples, columns=tissue_samples)
    linkage        = sch.linkage(sample_dists, method="ward")

    # Annotation bar colors for heatmap
    annot_cols     = meta_tissue[["diet", "training"]].iloc[::-1]   # tissue last, diet first
    col_colors     = pd.DataFrame({
        col: annot_cols[col].map(annotation_colors[col])
        for col in annot_cols.columns
    })

    g = sns.clustermap(
        dist_df,
        row_linkage=linkage, col_linkage=linkage,
        col_colors=col_colors,
        cmap="Blues_r",
        linewidths=0,
        xticklabels=False, yticklabels=False,
        figsize=(7, 5),
    )
    g.ax_heatmap.set_title(f"{tissue} — sample distance (VST)")
    g.savefig(f"{tissue}_dist_cluster_protein_coding.pdf", bbox_inches="tight")
    plt.show()
    plt.close("all")
    print(f"  Saved: {tissue}_dist_cluster_protein_coding.pdf")

    # PCA on top 500 most variable genes
    # Restricting to variable genes focuses PCA on informative signal
    # rather than the large number of uniformly expressed genes.
    row_vars   = vst.var(axis=1)
    top500     = row_vars.nlargest(min(500, len(row_vars))).index
    pca_input  = vst.loc[top500].T.values   # samples x genes

    pca         = PCA(n_components=3)
    pca_coords  = pca.fit_transform(pca_input)
    percent_var = pca.explained_variance_ratio_ * 100

    pca_df = pd.DataFrame(
        pca_coords, index=tissue_samples,
        columns=["PC1", "PC2", "PC3"]
    ).join(meta_tissue)

    # 2D PCA plot colored by pheno_class
    fig, ax = plt.subplots(figsize=(5, 4))
    for pheno, grp in pca_df.groupby("pheno_class"):
        ax.scatter(grp["PC1"], grp["PC2"], label=pheno, s=50)
    ax.set_xlabel(f"PC1: {percent_var[0]:.0f}% variance")
    ax.set_ylabel(f"PC2: {percent_var[1]:.0f}% variance")
    ax.legend(fontsize=8)
    ax.set_title(f"{tissue} — PCA (top 500 variable genes)")
    for spine in ax.spines.values():
        spine.set_visible(True)
    plt.tight_layout()
    plt.show()

    # 3D PCA plot colored by training group
    # The 3D view can reveal structure not visible in the PC1/PC2 plane,
    # particularly when PC3 captures a meaningful biological axis.
    fig3d = plt.figure(figsize=(6, 5))
    ax3d  = fig3d.add_subplot(111, projection="3d")
    for training_val, grp in pca_df.groupby("training"):
        color = annotation_colors["training"][training_val]
        ax3d.scatter(grp["PC1"], grp["PC2"], grp["PC3"],
                     c=color, label=training_val, s=40)
    ax3d.set_xlabel(f"PC1: {percent_var[0]:.0f}%")
    ax3d.set_ylabel(f"PC2: {percent_var[1]:.0f}%")
    ax3d.set_zlabel(f"PC3: {percent_var[2]:.0f}%")
    ax3d.set_title(f"{tissue} — 3D PCA (training)")
    ax3d.legend(fontsize=8)
    fig3d.savefig(f"{tissue}_vsd_pca_pc1-3_protein_coding_top500_training.pdf",
                  bbox_inches="tight")
    plt.show()
    plt.close("all")

    # ── Stage 2: DESeq2 ───────────────────────────────────────────
    # pydeseq2 expects samples x genes (rows x columns) for counts,
    # and a metadata DataFrame with sample names as index.
    dds = DeseqDataSet(
        counts=cts_tissue.T,           # samples x genes
        metadata=meta_tissue,
        design_factors="pheno_class",
        ref_level=["pheno_class", "SC"],  # SC = reference; contrasts are vs. SC
        refit_cooks=True,
    )
    dds.deseq2()
    print(f"  DESeq2 fit complete")

    tissue_results = {}

    # ── Stage 3a: TC vs SC and SH vs SC (default model) ──────────
    for contrast in [("pheno_class", "TC", "SC"), ("pheno_class", "SH", "SC")]:
        contrast_name = f"{contrast[1]}_vs_{contrast[2]}"
        print(f"  Contrast: {contrast_name}")

        stat = DeseqStats(
            dds,
            contrast=list(contrast),
            alpha=0.05,
            # IHW approximation: weight hypotheses by mean expression.
            # True IHW (independent hypothesis weighting) is not implemented
            # in pydeseq2; use standard BH correction here and note the
            # difference. For full IHW, use rpy2 with the R IHW package.
        )
        stat.summary()
        stat.lfc_shrink(coeff=f"pheno_class_{contrast[1]}_vs_{contrast[2]}")

        res = stat.results_df.copy()

        # MA plot: log mean expression vs. log fold change.
        # Points are colored red if padj < 0.05.
        # LFC shrinkage pulls unreliable estimates for low-count genes
        # toward zero, making the MA plot more interpretable.
        fig, ax = plt.subplots(figsize=(5, 4))
        non_sig = res[res["padj"].isna() | (res["padj"] >= 0.05)]
        sig_res = res[(res["padj"] < 0.05) & res["padj"].notna()]
        ax.scatter(np.log10(non_sig["baseMean"] + 1), non_sig["log2FoldChange"],
                   s=1, alpha=0.3, color="grey", rasterized=True)
        ax.scatter(np.log10(sig_res["baseMean"] + 1), sig_res["log2FoldChange"],
                   s=2, alpha=0.6, color="red", rasterized=True)
        ax.axhline(0, color="black", linewidth=0.8)
        ax.set_ylim(-2, 2)
        ax.set_xlabel("log10(mean expression + 1)")
        ax.set_ylabel("log2 fold change (shrunken)")
        ax.set_title(f"{tissue} — {contrast_name} MA plot")
        plt.tight_layout()
        plt.show()

        # Write significant DE genes (padj < 0.05)
        sig = sig_res.reset_index().rename(columns={"index": "gene"})
        out_path = f"{tissue}_sig_{contrast_name}_0.05.txt"
        sig.to_csv(out_path, sep="\t", index=False)
        print(f"    Significant genes: {len(sig):,} — saved to {out_path}")
        tissue_results[contrast_name] = res

    # ── Stage 3b: TH vs SH (relevel to SH as reference) ─────────
    # This is the key therapeutic contrast: does exercise training
    # rescue the transcriptional effects of HFD?
    # DESeq2 requires a new reference level and re-running the Wald test.
    print("  Contrast: TH_vs_SH (releveled)")
    dds_relevel = DeseqDataSet(
        counts=cts_tissue.T,
        metadata=meta_tissue,
        design_factors="pheno_class",
        ref_level=["pheno_class", "SH"],   # SH = reference for this contrast
        refit_cooks=True,
    )
    dds_relevel.deseq2()

    stat_th_sh = DeseqStats(
        dds_relevel,
        contrast=["pheno_class", "TH", "SH"],
        alpha=0.05,
    )
    stat_th_sh.summary()
    stat_th_sh.lfc_shrink(coeff="pheno_class_TH_vs_SH")

    res_th_sh = stat_th_sh.results_df.copy()

    fig, ax = plt.subplots(figsize=(5, 4))
    non_sig = res_th_sh[res_th_sh["padj"].isna() | (res_th_sh["padj"] >= 0.05)]
    sig_res = res_th_sh[(res_th_sh["padj"] < 0.05) & res_th_sh["padj"].notna()]
    ax.scatter(np.log10(non_sig["baseMean"] + 1), non_sig["log2FoldChange"],
               s=1, alpha=0.3, color="grey", rasterized=True)
    ax.scatter(np.log10(sig_res["baseMean"] + 1), sig_res["log2FoldChange"],
               s=2, alpha=0.6, color="red", rasterized=True)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_ylim(-2, 2)
    ax.set_xlabel("log10(mean expression + 1)")
    ax.set_ylabel("log2 fold change (shrunken)")
    ax.set_title(f"{tissue} — TH_vs_SH MA plot")
    plt.tight_layout()
    plt.show()

    sig_th_sh = sig_res.reset_index().rename(columns={"index": "gene"})
    sig_th_sh.to_csv(f"{tissue}_sig_pheno_class_TH_vs_SH_0.05.txt",
                     sep="\t", index=False)
    print(f"    TH vs SH significant genes: {len(sig_th_sh):,}")
    tissue_results["TH_vs_SH"] = res_th_sh

    # ── Save DESeq2 object ────────────────────────────────────────
    import pickle
    with open(f"dds_{tissue}.pkl", "wb") as f:
        pickle.dump(dds, f)
    print(f"  Saved: dds_{tissue}.pkl")

    results_store[tissue] = tissue_results

print("\nAll tissues complete.")

## Summary of DE results

A quick overview of how many significant genes were found per tissue and contrast. This gives an initial sense of which contrasts and tissues show the strongest transcriptional responses to diet and exercise.

In [ ]:
rows = []
for tissue, contrasts in results_store.items():
    for contrast_name, res in contrasts.items():
        sig = res[(res["padj"] < 0.05) & res["padj"].notna()]
        n_up = (sig["log2FoldChange"] > 0).sum()
        n_dn = (sig["log2FoldChange"] < 0).sum()
        rows.append({"tissue": tissue, "contrast": contrast_name,
                     "n_sig": len(sig), "n_up": n_up, "n_down": n_dn})

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))